<a href="https://colab.research.google.com/github/yosungcho/yosungcho.github.io/blob/main/03192026Grad_Cam.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# ============================================================
#  Grad-CAM Heatmap Pipeline — Google Colab Version
#  Yosung / JoJo Retinal Multi-Disease Screening Project
# ============================================================
#
#  STEP 0: Mount Drive
#  from google.colab import drive
#  drive.mount('/content/drive')
#
#  STEP 1: Install dependencies
#  !pip install onnxruntime opencv-python-headless matplotlib tensorflow -q
#
#  STEP 2: Run
#  !python gradcam_retinal_colab.py
#
# ============================================================

import os
import math
import numpy as np
import cv2
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from pathlib import Path

# ──────────────────────────────────────────────────────────────
#  GOOGLE DRIVE PATHS
# ──────────────────────────────────────────────────────────────
DRIVE_ROOT = "/content/drive/MyDrive/Jojo CNN"
MODEL_DIR  = f"{DRIVE_ROOT}/Inference models"
IMAGE_DIR  = f"{DRIVE_ROOT}/Grad-CAM images"
OUTPUT_DIR = f"{DRIVE_ROOT}/Grad-CAM output"

# ──────────────────────────────────────────────────────────────
#  DISEASE CONFIG
# ──────────────────────────────────────────────────────────────
DISEASE_CONFIG = {
    "DR": {
        "model_file": "DiabeticRunModela.h5",
        "backend":    "keras",
        "label":      "Diabetic Retinopathy",
        "color":      "#FF4444",
    },
    "Glaucoma": {
        "model_file": "GlaucomaResNet18_v2.onnx",
        "backend":    "onnx",
        "label":      "Glaucoma",
        "color":      "#4488FF",
    },
    "Cataract": {
        "model_file": "CataractRunModela.h5",
        "backend":    "keras",
        "label":      "Cataract",
        "color":      "#44BB44",
    },
    "HR": {
        "model_file": "HypertensiveResNet18_v6.onnx",
        "backend":    "onnx",
        "label":      "Hypertensive Retinopathy",
        "color":      "#FF8800",
    },
}

# ──────────────────────────────────────────────────────────────
#  GLOBAL PREPROCESSING CONFIG
# ──────────────────────────────────────────────────────────────
DISPLAY_SIZE         = 224    # fixed size used for overlay display output
GAUSSIAN_BLUR_KERNEL = 0      # set to 5 if blur was applied during training
NORMALIZE_MEAN       = np.array([0.485, 0.456, 0.406], dtype=np.float32)
NORMALIZE_STD        = np.array([0.229, 0.224, 0.225], dtype=np.float32)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# ──────────────────────────────────────────────────────────────
#  HELPER: probe Keras model to find working input size
# ──────────────────────────────────────────────────────────────

def probe_keras_input_size(model):
    """
    Tries common image sizes until the model accepts input without error.
    This handles Sequential models saved without explicit input_shape.
    """
    candidates = [60, 64, 80, 96, 100, 112, 120, 124, 128, 150, 160, 224]
    for size in candidates:
        try:
            dummy = np.zeros((1, size, size, 3), dtype=np.float32)
            model(dummy, training=False)
            print(f"  Probed working input size: {size}x{size}")
            return size
        except Exception:
            continue
    raise ValueError(
        "Cannot find working input size for this Keras model. "
        "Check your .h5 file integrity."
    )


# ──────────────────────────────────────────────────────────────
#  PREPROCESSING  (model_size may differ from DISPLAY_SIZE)
# ──────────────────────────────────────────────────────────────

def preprocess(image_path, model_size):
    """
    Returns:
        img_float : [DISPLAY_SIZE, DISPLAY_SIZE, 3] float32 0-1  (for overlay)
        img_chw   : [1, 3, model_size, model_size] normalized     (ONNX)
        img_hwc   : [1, model_size, model_size, 3] 0-1            (Keras)
    """
    img_bgr = cv2.imread(str(image_path))
    if img_bgr is None:
        raise FileNotFoundError(f"Cannot read: {image_path}")

    # Display copy at fixed 224px
    disp      = cv2.resize(img_bgr, (DISPLAY_SIZE, DISPLAY_SIZE))
    img_float = cv2.cvtColor(disp, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    # Model copy at the model's actual training resolution
    model_img = cv2.resize(img_bgr, (model_size, model_size))
    if GAUSSIAN_BLUR_KERNEL > 0:
        model_img = cv2.GaussianBlur(
            model_img, (GAUSSIAN_BLUR_KERNEL, GAUSSIAN_BLUR_KERNEL), 0)
    model_rgb = cv2.cvtColor(model_img, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0

    img_norm = (model_rgb - NORMALIZE_MEAN) / NORMALIZE_STD
    img_chw  = img_norm.transpose(2, 0, 1)[np.newaxis]   # [1,3,H,W]
    img_hwc  = model_rgb[np.newaxis]                      # [1,H,W,3]
    return img_float, img_chw, img_hwc


# ──────────────────────────────────────────────────────────────
#  ONNX GRAD-CAM  (occlusion sensitivity)
# ──────────────────────────────────────────────────────────────

class OnnxGradCAM:
    def __init__(self, model_path):
        import onnxruntime as ort
        self.sess = ort.InferenceSession(
            model_path,
            providers=["CUDAExecutionProvider", "CPUExecutionProvider"])
        inp = self.sess.get_inputs()[0]
        self.input_name  = inp.name
        self.output_name = self.sess.get_outputs()[0].name
        shape = inp.shape
        self.model_size  = int(shape[2]) if isinstance(shape[2], int) else DISPLAY_SIZE
        print(f"  ONNX input : {self.input_name} {shape}")
        print(f"  ONNX output: {self.output_name}")
        print(f"  Model size : {self.model_size}x{self.model_size}")

    def _infer(self, img_chw):
        return self.sess.run(
            [self.output_name],
            {self.input_name: img_chw.astype(np.float32)})[0]

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-float(x)))

    def generate(self, img_chw):
        size  = img_chw.shape[2]
        base  = self._sigmoid(self._infer(img_chw).flatten()[0])
        grid  = 14
        cell  = max(1, size // grid)
        imp   = np.zeros((grid, grid), dtype=np.float32)
        for i in range(grid):
            for j in range(grid):
                occ = img_chw.copy()
                r0, r1 = i*cell, min((i+1)*cell, size)
                c0, c1 = j*cell, min((j+1)*cell, size)
                occ[0, :, r0:r1, c0:c1] = 0.0
                imp[i, j] = max(0.0, base - self._sigmoid(self._infer(occ).flatten()[0]))
        heatmap = cv2.resize(imp, (DISPLAY_SIZE, DISPLAY_SIZE))
        if heatmap.max() > 0:
            heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)
        return heatmap, base


# ──────────────────────────────────────────────────────────────
#  KERAS GRAD-CAM  (true gradient-based via GradientTape)
# ──────────────────────────────────────────────────────────────

class KerasGradCAM:
    """
    Grad-CAM for Keras Sequential models using a layer-split + GradientTape strategy.

    Standard approach (tf.keras.Model with multiple outputs) fails on Sequential
    models saved before being called — layer.output tensors are undefined.
    Instead: split layers at the last Conv2D, run both halves inside GradientTape,
    differentiate predictions w.r.t. the intermediate feature maps directly.
    No tf.keras.Model() rebuild needed — avoids the input_layer naming collision.
    """

    def __init__(self, model_path):
        import tensorflow as tf
        self.tf = tf

        print(f"  Loading Keras model...")
        self.model = tf.keras.models.load_model(model_path, compile=False)

        # Step 1: detect working input size via probing
        self.model_size = probe_keras_input_size(self.model)

        # Step 2: warm up so all layer weights are fully loaded
        dummy = np.zeros((1, self.model_size, self.model_size, 3), dtype=np.float32)
        _ = self.model(dummy, training=False)

        # Step 3: flatten all layers (including nested Sequential blocks)
        flat_layers = []
        for layer in self.model.layers:
            if hasattr(layer, 'layers'):
                flat_layers.extend(layer.layers)
            else:
                flat_layers.append(layer)

        # Step 4: find last Conv2D
        last_conv_idx = None
        for i, layer in enumerate(flat_layers):
            if isinstance(layer, tf.keras.layers.Conv2D):
                last_conv_idx = i

        if last_conv_idx is None:
            raise ValueError("No Conv2D layer found in Keras model")

        self.last_conv_name = flat_layers[last_conv_idx].name
        print(f"  Last Conv2D: {self.last_conv_name} "
              f"(layer {last_conv_idx}/{len(flat_layers)-1})")

        # Step 5: split into two sequential layer lists
        self.layers_a = flat_layers[:last_conv_idx + 1]   # input → feature maps
        self.layers_b = flat_layers[last_conv_idx + 1:]   # feature maps → prediction
        print(f"  Split: {len(self.layers_a)} layers to conv, "
              f"{len(self.layers_b)} layers to output")

    def _forward(self, layers, x):
        """Pass x through a list of layers, skipping InputLayer."""
        for layer in layers:
            if isinstance(layer, self.tf.keras.layers.InputLayer):
                continue
            x = layer(x, training=False)
        return x

    def _sigmoid(self, x):
        return 1.0 / (1.0 + np.exp(-float(x)))

    def generate(self, img_hwc):
        """
        img_hwc: [1, model_size, model_size, 3] float32 in [0,1]
        Returns: heatmap [DISPLAY_SIZE, DISPLAY_SIZE] in [0,1], confidence float
        """
        tf         = self.tf
        img_tensor = tf.cast(img_hwc, tf.float32)

        with tf.GradientTape() as tape:
            # Run part A to get conv feature maps, then watch them
            conv_output = self._forward(self.layers_a, img_tensor)
            tape.watch(conv_output)
            # Run part B to get predictions
            preds = self._forward(self.layers_b, conv_output)
            pred  = preds[0]
            score = pred[0] if pred.shape[-1] == 1 else tf.reduce_max(pred)

        grads   = tape.gradient(score, conv_output)
        pooled  = tf.reduce_mean(grads, axis=(0, 1, 2))
        cam     = tf.reduce_sum(
            tf.multiply(pooled, conv_output[0]), axis=-1).numpy()
        cam     = np.maximum(cam, 0)

        heatmap = cv2.resize(cam, (DISPLAY_SIZE, DISPLAY_SIZE))
        if heatmap.max() > 0:
            heatmap = (heatmap - heatmap.min()) / (heatmap.max() - heatmap.min() + 1e-8)

        raw        = preds.numpy().flatten()[0]
        confidence = self._sigmoid(raw) if preds.shape[-1] == 1                      else float(np.max(preds.numpy()))
        return heatmap, confidence


# ──────────────────────────────────────────────────────────────
#  OVERLAY + FIGURES
# ──────────────────────────────────────────────────────────────

def make_overlay(img_float, heatmap, alpha=0.45):
    col = cv2.applyColorMap(np.uint8(255 * heatmap), cv2.COLORMAP_JET)
    rgb = cv2.cvtColor(col, cv2.COLOR_BGR2RGB) / 255.0
    return np.clip((1 - alpha) * img_float + alpha * rgb, 0, 1)


def save_single_figure(image_path, img_float, heatmap, overlay,
                       disease_label, confidence, out_path):
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.patch.set_facecolor("#1a1a2e")
    for ax, (img, title) in zip(axes, [
        (img_float,                      "Original fundus image"),
        (plt.cm.jet(heatmap)[:, :, :3], "Grad-CAM heatmap"),
        (overlay,                        "Overlay"),
    ]):
        ax.imshow(img); ax.set_title(title, color="white", fontsize=11, pad=6); ax.axis("off")
    fig.suptitle(
        f"{disease_label}  |  {Path(image_path).stem}  |  Confidence: {confidence:.1%}",
        color="white", fontsize=13, fontweight="bold", y=1.01)
    plt.tight_layout()
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"    Saved → {Path(out_path).name}")


def save_summary_grid(results, out_path, max_per_disease=5):
    diseases = list(dict.fromkeys(r["disease"] for r in results))
    n_cols   = min(max_per_disease,
                   max(sum(1 for r in results if r["disease"] == d) for d in diseases))
    fig = plt.figure(figsize=(n_cols * 3.5, len(diseases) * 3.5))
    fig.patch.set_facecolor("#0f0f1a")
    gs  = gridspec.GridSpec(len(diseases), n_cols, figure=fig, hspace=0.4, wspace=0.1)
    for row, disease in enumerate(diseases):
        drows = [r for r in results if r["disease"] == disease][:max_per_disease]
        for col, res in enumerate(drows):
            ax = fig.add_subplot(gs[row, col])
            ax.imshow(res["overlay"])
            ax.set_title(f"{res['confidence']:.0%}", color="#aaffaa", fontsize=9)
            ax.axis("off")
            if col == 0:
                ax.set_ylabel(DISEASE_CONFIG[disease]["label"],
                              color="white", fontsize=9,
                              fontweight="bold", rotation=90, labelpad=8)
    fig.suptitle("Grad-CAM heatmap analysis — retinal multi-disease screening",
                 color="white", fontsize=13, fontweight="bold", y=1.01)
    plt.savefig(out_path, dpi=150, bbox_inches="tight", facecolor=fig.get_facecolor())
    plt.close(fig)
    print(f"\n✓ Summary grid → {Path(out_path).name}")


# ──────────────────────────────────────────────────────────────
#  AUTO-INSTALL + MAIN
# ──────────────────────────────────────────────────────────────

def _check_deps():
    import importlib, subprocess, sys
    for pkg in ["onnxruntime", "tensorflow"]:
        if importlib.util.find_spec(pkg) is None:
            print(f"  Installing {pkg}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])


def main():
    _check_deps()
    IMG_EXTS    = {".jpg", ".jpeg", ".png", ".bmp"}
    all_results = []

    for disease_key, cfg in DISEASE_CONFIG.items():
        model_path   = os.path.join(MODEL_DIR, cfg["model_file"])
        disease_imgs = os.path.join(IMAGE_DIR, disease_key)
        disease_out  = os.path.join(OUTPUT_DIR, disease_key)

        print(f"\n{'─'*55}")
        print(f"  {cfg['label']}")
        print(f"  Model  : {cfg['model_file']}")
        print(f"  Backend: {cfg['backend'].upper()}")

        if not os.path.isfile(model_path):
            print(f"  [SKIP] Model not found: {model_path}"); continue
        if not os.path.isdir(disease_imgs):
            print(f"  [SKIP] Image folder not found: {disease_imgs}"); continue

        os.makedirs(disease_out, exist_ok=True)

        try:
            if cfg["backend"] == "onnx":
                cam_model  = OnnxGradCAM(model_path)
            else:
                cam_model  = KerasGradCAM(model_path)
            model_size = cam_model.model_size
        except Exception as e:
            print(f"  [ERROR] Loading model: {e}"); continue

        images = sorted([
            f for f in os.listdir(disease_imgs)
            if Path(f).suffix.lower() in IMG_EXTS
        ])[:10]
        print(f"  Images : {len(images)}  |  Input size: {model_size}x{model_size}")

        for img_file in images:
            img_path = os.path.join(disease_imgs, img_file)
            try:
                img_float, img_chw, img_hwc = preprocess(img_path, model_size)
                if cfg["backend"] == "onnx":
                    heatmap, conf = cam_model.generate(img_chw)
                else:
                    heatmap, conf = cam_model.generate(img_hwc)
                overlay  = make_overlay(img_float, heatmap)
                out_file = os.path.join(disease_out, f"gradcam_{Path(img_file).stem}.jpg")
                save_single_figure(img_path, img_float, heatmap, overlay,
                                   cfg["label"], conf, out_file)
                all_results.append({"disease": disease_key,
                                    "overlay": overlay, "confidence": conf})
            except Exception as e:
                print(f"    [ERROR] {img_file}: {e}")

    if all_results:
        save_summary_grid(all_results, os.path.join(OUTPUT_DIR, "gradcam_summary_grid.jpg"))

    print(f"\n{'='*55}")
    print(f"✓ All outputs saved to:\n  {OUTPUT_DIR}")


if __name__ == "__main__":
    main()



───────────────────────────────────────────────────────
  Diabetic Retinopathy
  Model  : DiabeticRunModela.h5
  Backend: KERAS
  Loading Keras model...
  Probed working input size: 128x128
  Last Conv2D: conv2d_1 (layer 2/7)
  Split: 3 layers to conv, 5 layers to output
  Images : 3  |  Input size: 128x128
    Saved → gradcam_D1 copy.jpg
    Saved → gradcam_D2 copy.jpg
    Saved → gradcam_D3 copy.jpg

───────────────────────────────────────────────────────
  Glaucoma
  Model  : GlaucomaResNet18_v2.onnx
  Backend: ONNX
  ONNX input : input [1, 3, 224, 224]
  ONNX output: output
  Model size : 224x224
  Images : 5  |  Input size: 224x224


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


    Saved → gradcam_G1 copy.jpg
    Saved → gradcam_G2 copy.jpg
    Saved → gradcam_G3 copy.jpg
    Saved → gradcam_G4 copy.jpg
    Saved → gradcam_G5 copy.jpg

───────────────────────────────────────────────────────
  Cataract
  Model  : CataractRunModela.h5
  Backend: KERAS
  Loading Keras model...
  Probed working input size: 128x128
  Last Conv2D: conv2d_3 (layer 2/7)
  Split: 3 layers to conv, 5 layers to output
  Images : 5  |  Input size: 128x128
    Saved → gradcam_C1 copy.jpg
    Saved → gradcam_C2 copy.jpg
    Saved → gradcam_C3 copy.jpg
    Saved → gradcam_C4 copy.jpg
    Saved → gradcam_C5 copy.jpg

───────────────────────────────────────────────────────
  Hypertensive Retinopathy
  Model  : HypertensiveResNet18_v6.onnx
  Backend: ONNX


/usr/local/lib/python3.12/dist-packages/onnxruntime/capi/onnxruntime_inference_collection.py:123: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


  ONNX input : input [1, 3, 224, 224]
  ONNX output: output
  Model size : 224x224
  Images : 3  |  Input size: 224x224
    Saved → gradcam_H1.jpg
    Saved → gradcam_H2.jpg
    Saved → gradcam_H3.jpg

✓ Summary grid → gradcam_summary_grid.jpg

✓ All outputs saved to:
  /content/drive/MyDrive/Jojo CNN/Grad-CAM output
